# TTM-Small-4M × EmbeddingGemma 2 — tích hợp trọng số thật (CPU, một lượt)

**Mục tiêu:** ánh xạ tokenizer TRAIN-only 2.048 ID sang bảng input EmbeddingGemma 2 512 chiều; adapter 512→256; nạp checkpoint P02 bước 1.024; kiểm forward và sinh candidate riêng. **Không RAG, không huấn luyện dài, không đọc TEST, không kết luận khoa học.**

Chạy trên **Colab CPU**; không cần Tesla T4. Notebook tự tải source ZIP từ GitHub commit cố định. Không cần bạn tải ZIP từ Drive. Nếu chưa có `T1_BPE2048_REAL_TRAIN_MANIFEST.json` trên Drive, chỉ upload **một tệp manifest nhỏ gốc** từ thư mục `_tmp/ttm_t1_real_source_v1` của repo Windows khi được hỏi. Checkpoint TTM bước 1.024 sẽ được dùng trực tiếp từ Drive.

Nếu Colab cache EmbeddingGemma 2 đã bị xóa, lần chạy này cần tải lại 4 tệp ghim phiên bản (trong đó trọng số khoảng 1,1 GB); không tải lại TinyStories hoặc Qwen.
Sau khi tạo candidate, cùng lượt chạy sẽ đối chiếu logits baseline/untied, nạp lại candidate, kiểm nhân quả và ghi `PROOF_READBACK.json`.

**Sửa đường dẫn checkpoint (2026-11-09):** thư mục thực tế trên Drive là `MyDrive/TTM/ttm-small-pilot/...`, không phải `MyDrive/ttm-small-pilot/...`. SHA256 giữ nguyên, không thay model/source, không rerun science.


In [ ]:
%pip -q install transformers==5.22.1 huggingface-hub safetensors


In [ ]:
from pathlib import Path
import hashlib, importlib.util, shutil, sys, urllib.request, zipfile
from google.colab import drive, files
from huggingface_hub import snapshot_download

# One bounded engineering integration, CPU only; never reads TRAIN/VAL/TEST.
SOURCE_COMMIT = "efc4eef884dc35fe972ff9124caa20166aefcf7e"
ZIP_SHA256 = "bce8ffa8bc954d7f1ba87b08c033e6e8c3dbc925be39cc4b78ef90b6d66a026e"
SOURCE_SHA256 = "d11da61e9d7bb040f3263e680f026817a513553c186ab243e184ffaf6e68cd39"
P02_SHA256 = "e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07"
MANIFEST_SHA256 = "cba18e60cee3d08a353ee6d4a451017289e45af7b4b1179c87171b9b78d7dfdc"
CHECKPOINT_SHA256 = "cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b"
DONOR_REV = "f9972ac22e696a105c482202646979e33e4b5a1e"
DONOR_PINS = {
 "model.safetensors": "197a32965d4b1105faf060417baa899e193fb73cd401f42ec9295234d5553d79",
 "tokenizer.json": "4d777ef5bdc1aa36227abdfb77c3e49e7b9c892d16e1b6bda41c393504828be4",
 "tokenizer_config.json": "17bd5d6e9364ca49a534e1502076593317c298d4a663623091ed45388f004874",
 "config.json": "b8f1e9931b57fbc054acdb445c41765d55b0074c58d145fa82839941ad1b5bb3"
}
def sha(path):
 h=hashlib.sha256()
 with open(path,"rb") as stream:
  for block in iter(lambda:stream.read(1024*1024),b""):h.update(block)
 return h.hexdigest()
def require(path,expected,label):
 if not path.is_file() or sha(path)!=expected:raise RuntimeError("STOP_EXACT_SHA_MISMATCH_OR_MISSING:"+label)
 return path

drive.mount("/content/drive")
drive_root=Path("/content/drive/MyDrive")
ckpt=drive_root/"TTM"/"ttm-small-pilot"/"t4_science_4m_1024_a2_20261108_v1"/"checkpoints"/"step_1024"/"checkpoint.pt"
require(ckpt,CHECKPOINT_SHA256,"TRAINED_STEP1024_CHECKPOINT")
inputs=drive_root/"TTM_Small4M_EmbeddingSwap_Input"
manifest=inputs/"T1_BPE2048_REAL_TRAIN_MANIFEST.json"
if not manifest.is_file():
 print("TTM_SWAP_NEED_INPUT=UPLOAD_ORIGINAL_T1_BPE_MANIFEST_ONLY")
 print(r"Windows: D:\WORK\RESEARCH\TokenToModel-Lab\_tmp\ttm_t1_real_source_v1\T1_BPE2048_REAL_TRAIN_MANIFEST.json")
 uploaded=files.upload()
 if set(uploaded)!={manifest.name}:raise RuntimeError("STOP_WRONG_OR_MISSING_BPE_MANIFEST")
 if hashlib.sha256(uploaded[manifest.name]).hexdigest()!=MANIFEST_SHA256:
  raise RuntimeError("STOP_BPE_MANIFEST_HASH_MISMATCH")
 inputs.mkdir(parents=True,exist_ok=True)
 manifest.write_bytes(uploaded[manifest.name])
require(manifest,MANIFEST_SHA256,"TRAIN_ONLY_BPE2048")

local=Path("/content/ttm_embedding_swap")
local.mkdir(parents=True,exist_ok=True)
source_url=("https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/"
            +SOURCE_COMMIT+"/tools/embeddinggemma2_swap/source_bundle_20261109.zip")
bundle=local/"source_bundle_20261109.zip"
if not bundle.is_file():
 with urllib.request.urlopen(source_url,timeout=90) as response:raw=response.read()
 if hashlib.sha256(raw).hexdigest()!=ZIP_SHA256:raise RuntimeError("STOP_SOURCE_BUNDLE_SHA256")
 bundle.write_bytes(raw)
require(bundle,ZIP_SHA256,"FROZEN_SOURCE_ZIP")
with zipfile.ZipFile(bundle) as z:
 for name,expect in (("ttm_embeddinggemma2_swap.py",SOURCE_SHA256),("p02_exact.py",P02_SHA256)):
  raw=z.read(name)
  if hashlib.sha256(raw).hexdigest()!=expect:raise RuntimeError("STOP_SOURCE_ENTRY_SHA256:"+name)
  (local/name).write_bytes(raw)
spec=importlib.util.spec_from_file_location("ttm_swap_real",local/"ttm_embeddinggemma2_swap.py")
swap=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=swap
spec.loader.exec_module(swap)
require(local/"ttm_embeddinggemma2_swap.py",SOURCE_SHA256,"SWAP_PY")

# Reuse an existing pinned cache if Colab still has one. If absent,
# download just four pinned files (not multimodal full model execution).
cache_dirs=[Path("/content/ttm_embeddinggemma2_d1_preflight_st610_v1/hf_cache"),
            Path("/content/ttm_small_swap_hf")]
donor=None
for cache in cache_dirs:
 p=cache/"models--google--embeddinggemma-2"/"snapshots"/DONOR_REV
 if all((p/name).is_file() and sha(p/name)==digest for name,digest in DONOR_PINS.items()):
  donor=p;break
if donor is None:
 if shutil.disk_usage("/content").free < 3*1024**3:
  raise RuntimeError("STOP_INSUFFICIENT_LOCAL_DISK_3GIB")
 donor=Path(snapshot_download(
  repo_id="google/embeddinggemma-2",revision=DONOR_REV,
  allow_patterns=list(DONOR_PINS),cache_dir="/content/ttm_small_swap_hf",max_workers=3))
for name,digest in DONOR_PINS.items():require(donor/name,digest,"EMBEDDINGGEMMA2_"+name)

# Output namespace is separate. Any failed source verification occurs before writes.
output=drive_root/"TTM_Small4M_EmbeddingSwap"/"real_token_mapping_step1024_once_20261109_v1"
if output.exists():raise RuntimeError("STOP_EXISTING_CANDIDATE_NO_OVERWRITE")
print("TTM_SWAP_PINNED_INPUTS=PASS")
receipt=swap.real_build(local/"p02_exact.py",manifest,donor,ckpt,output)
print("TTM_SWAP_REAL_STATUS="+receipt["status"])
print("TTM_SWAP_MAPPING="+str(receipt["mapping"]))
print("TTM_SWAP_RESULT="+str(output/"PROOF_RESULT.json"))

# Independent readback: actual trained checkpoint and transferred candidate,
# no optimizer step and no protected VAL/TEST data.
import copy, json, torch
p02=swap.load_p02(local/"p02_exact.py")
ckpt_payload=torch.load(ckpt,map_location="cpu",weights_only=True)
base=p02.GPT(swap.p02_config(p02)).float().eval()
base.load_state_dict(ckpt_payload["model"],strict=True)
untied=swap.control_untied(base).eval()
x=torch.tensor([[1,42,110,500,256,300],[2,29,340,4,10,60]],dtype=torch.long)
with torch.no_grad():
 baseline_logits=base(x)[0]
 untied_logits=untied(x)[0]
torch.testing.assert_close(baseline_logits,untied_logits,rtol=0,atol=0)
artifact=torch.load(output/"candidate_initialization.pt",map_location="cpu",weights_only=True)
state=artifact["model_state_dict"]
candidate=swap.control_untied(base).eval()
candidate.tok=swap.MappedGemmaInput(
 state["tok.source.weight"],state["tok.valid"],
 state["tok.fallback"],state["tok.proj.weight"])
candidate.load_state_dict(state,strict=True)
candidate.eval()
with torch.no_grad():
 candidate_logits=candidate(x)[0]
 changed=x.clone()
 changed[:,-1]=123
 earlier_candidate=candidate(changed)[0]
if tuple(candidate_logits.shape)!=(2,6,2048) or not bool(torch.isfinite(candidate_logits).all()):
 raise RuntimeError("CANDIDATE_LOGITS_INVALID")
torch.testing.assert_close(candidate_logits[:,:-1],earlier_candidate[:,:-1],atol=0,rtol=0)
max_abs_delta=float((candidate_logits-baseline_logits).abs().max())
checks={
 "status":"PASS_REAL_CHECKPOINT_READBACK_NO_SCIENCE",
 "baseline_vs_untied_logits_exact":True,
 "candidate_logits_finite":True,
 "candidate_logit_shape":[2,6,2048],
 "candidate_minus_baseline_max_abs_logit":max_abs_delta,
 "causal_future_token_isolation":True,
 "mapped_vocab":int(state["tok.valid"].sum()),
 "vocab_size":2048,
 "donor_embedding_width":512,
 "ttm_embedding_width":256,
 "trained_checkpoint_sha256":CHECKPOINT_SHA256,
 "source_bundle_sha256":ZIP_SHA256,
 "train_run_reexecuted":False,
 "science_executed":False,
 "validation_read":False,
 "test_read":False,
 "gpu_required":False
}
with (output/"PROOF_READBACK.json").open("x",encoding="utf-8") as handle:
 json.dump(checks,handle,indent=2,sort_keys=True)
 handle.write("\n")
print("TTM_SWAP_READBACK_STATUS="+checks["status"])
print("TTM_SWAP_LOGIT_DELTA_MAX="+str(max_abs_delta))
print("TTM_SWAP_READBACK_RESULT="+str(output/"PROOF_READBACK.json"))
